# Step 1: Identify Tour Cities

In [1]:
import pandas as pd
import numpy as np
import argparse

In [2]:
def min_max_normalization(series):
    if series.max() == series.min():
        return series * 0
    return (series - series.min()) / (series.max() - series.min())

In [3]:
## get the input data
artist_data = pd.read_csv('data/artist.csv')
artist_data

,artist_id,artist_name,primary_genre,career_stage,us_monthly_listeners
0,A001,Nova Reyes,indie pop,mid-level,4200000
1,A002,The Copper Pines,folk rock,emerging,900000
2,A003,Kairo Blaze,hip hop,established,11000000


In [4]:
## get the input data
metrics_data = pd.read_csv('data/city_fan_metrics.csv')
metrics_data

,artist_id,city_id,city,state,spotify_monthly_listeners,youtube_city_affinity,streams_28d,instagram_likers,instagram_followers,youtube_viewers_28d,product_sales_units,song_sales_units
0,A001,C001,New York,NY,523829,0.419,5538052,25390,48617,239339,2147,3132
1,A001,C002,Los Angeles,CA,479750,0.532,4593737,38798,55733,193626,2954,2326
2,A001,C003,Chicago,IL,610530,0.888,5326103,67523,65913,260617,5755,4053
3,A001,C004,Dallas,TX,239295,0.415,2857535,14559,24704,97167,1255,1760
4,A001,C005,Houston,TX,175846,0.309,1800954,11265,20604,72811,715,1080
...,...,...,...,...,...,...,...,...,...,...,...,...
115,A003,C036,Detroit,MI,571590,0.628,5955099,51103,55671,287276,5100,3499
116,A003,C037,St. Louis,MO,136894,0.254,1385170,6364,15849,52423,398,839
117,A003,C038,Milwaukee,WI,233868,0.945,2261699,22588,26091,110857,2117,1811
118,A003,C039,Richmond,VA,78099,0.279,736577,3770,9060,41239,261,450


In [5]:
## get the input data
cities_data = pd.read_csv('data/cities.csv')
cities_data.head()

,city_id,city,state,lat,lon,metro_population
0,C001,New York,NY,40.7128,-74.0060,19500000
1,C002,Los Angeles,CA,34.0522,-118.2437,12800000
2,C003,Chicago,IL,41.8781,-87.6298,9300000
3,C004,Dallas,TX,32.7767,-96.7970,8100000
4,C005,Houston,TX,29.7604,-95.3698,7500000


In [6]:
metrics_data = metrics_data.merge(cities_data,on=['city_id','city','state'],how='left')
metrics_data

,artist_id,city_id,city,state,spotify_monthly_listeners,youtube_city_affinity,streams_28d,instagram_likers,instagram_followers,youtube_viewers_28d,product_sales_units,song_sales_units,lat,lon,metro_population
0,A001,C001,New York,NY,523829,0.419,5538052,25390,48617,239339,2147,3132,40.7128,-74.0060,19500000
1,A001,C002,Los Angeles,CA,479750,0.532,4593737,38798,55733,193626,2954,2326,34.0522,-118.2437,12800000
2,A001,C003,Chicago,IL,610530,0.888,5326103,67523,65913,260617,5755,4053,41.8781,-87.6298,9300000
3,A001,C004,Dallas,TX,239295,0.415,2857535,14559,24704,97167,1255,1760,32.7767,-96.7970,8100000
4,A001,C005,Houston,TX,175846,0.309,1800954,11265,20604,72811,715,1080,29.7604,-95.3698,7500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,A003,C036,Detroit,MI,571590,0.628,5955099,51103,55671,287276,5100,3499,42.3314,-83.0458,4300000
116,A003,C037,St. Louis,MO,136894,0.254,1385170,6364,15849,52423,398,839,38.6270,-90.1994,2800000
117,A003,C038,Milwaukee,WI,233868,0.945,2261699,22588,26091,110857,2117,1811,43.0389,-87.9065,1600000
118,A003,C039,Richmond,VA,78099,0.279,736577,3770,9060,41239,261,450,37.5407,-77.4360,1300000


In [7]:
## set the weights corresponding to each factor and variables
factor_weights = {
    'Passive Engagement': 0.35,
    'Proactive Engagement': 0.65
}

variable_weights = {
    'Passive Engagement': {
        'spotify_monthly_listeners_score': 0.45,
        'youtube_city_affinity_score': 0.25,
        'streams_28d_score': 0.3
    },
    'Proactive Engagement': {
        'instagram_likers_score': 0.1,
        'instagram_followers_score': 0.2,
        'youtube_viewers_28d_score': 0.15,
        'product_sales_units_score': 0.35,
        'song_sales_units_score': 0.2
    }
}




In [8]:
def build_composite_index(input_df, variable_weights, factor_weights):

    # -- Step 1: scale each variable
    def penetration_rate(metric_series, population_series):
        return metric_series / population_series

    def log_volume(metric_series):
        return np.log1p(metric_series)

    count_cols = ['spotify_monthly_listeners', 'streams_28d', 'instagram_likers',
        'instagram_followers', 'youtube_viewers_28d', 'product_sales_units',
        'song_sales_units']

    output = input_df[['artist_id', 'city_id', 'city', 'state','lat','lon']].copy()

    # -- volume branch: log1p, then min-max scaled per artist
    log_cols = [col + '_log' for col in count_cols]
    for col, log_col in zip(count_cols, log_cols):
        output[log_col] = log_volume(input_df[col])
    output[log_cols] = output.groupby('artist_id')[log_cols].transform(min_max_normalization)

    # -- penetration branch: raw / metro_population, then min-max scaled per artist
    penetration_cols = [col + '_penetration' for col in count_cols]
    for col, pen_col in zip(count_cols, penetration_cols):
        output[pen_col] = penetration_rate(input_df[col], input_df['metro_population'])
    output[penetration_cols] = output.groupby('artist_id')[penetration_cols].transform(min_max_normalization)

    # -- blend volume + penetration 50/50 into one score per count metric
    score_cols = [col + '_score' for col in count_cols]
    output[score_cols] = (
        0.5 * output[log_cols].values
        + 0.5 * output[penetration_cols].values
    )

    # -- youtube_city_affinity is already an affinity index: skip log/per-capita, just min-max
    output['youtube_city_affinity_score'] = input_df.groupby('artist_id')['youtube_city_affinity'].transform(min_max_normalization)

    # -- Step 2: weighted blend within each factor
    passive_weights = pd.Series(variable_weights['Passive Engagement'])
    weighted_cols = output[passive_weights.index].mul(passive_weights, axis=1)
    output['passive_score'] = weighted_cols.sum(axis=1)

    proactive_weights = pd.Series(variable_weights['Proactive Engagement'])
    weighted_cols = output[proactive_weights.index].mul(proactive_weights, axis=1)
    output['proactive_score'] = weighted_cols.sum(axis=1)

    output['city_index'] = 100 * (
        factor_weights['Passive Engagement'] * output['passive_score']
        + factor_weights['Proactive Engagement'] * output['proactive_score']
    )

    return output

In [9]:
index_output = build_composite_index(metrics_data, variable_weights, factor_weights)

# rank
index_output['rank'] = index_output.groupby('artist_id')['city_index'].rank(ascending=False)
index_output

,artist_id,city_id,city,state,lat,lon,spotify_monthly_listeners_log,streams_28d_log,instagram_likers_log,instagram_followers_log,...,instagram_likers_score,instagram_followers_score,youtube_viewers_28d_score,product_sales_units_score,song_sales_units_score,youtube_city_affinity_score,passive_score,proactive_score,city_index,rank
0,A001,C001,New York,NY,40.7128,-74.0060,0.951107,1.000000,0.747470,0.901367,...,0.402155,0.502084,0.578478,0.434018,0.558545,0.289731,0.498772,0.491019,49.373287,9.0
1,A001,C002,Los Angeles,CA,34.0522,-118.2437,0.923047,0.939830,0.856941,0.945634,...,0.532011,0.653800,0.589570,0.506842,0.537611,0.427873,0.564505,0.557314,55.983047,7.0
2,A001,C003,Chicago,IL,41.8781,-87.6298,1.000000,0.987440,1.000000,1.000000,...,0.787293,0.871307,0.845608,0.705838,0.907735,0.863081,0.824424,0.808422,81.402265,2.0
3,A001,C004,Dallas,TX,32.7767,-96.7970,0.701005,0.787036,0.603888,0.681975,...,0.351885,0.431152,0.426130,0.396882,0.535806,0.284841,0.431617,0.431408,43.148108,12.0
4,A001,C005,Houston,TX,29.7604,-95.3698,0.602656,0.638455,0.537666,0.623165,...,0.305940,0.380671,0.342018,0.321280,0.379731,0.155257,0.313365,0.346425,33.485421,19.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,A003,C036,Detroit,MI,42.3314,-83.0458,0.651180,0.713836,0.723969,0.586073,...,0.624576,0.548179,0.613675,0.589865,0.605321,0.543038,0.593235,0.591662,59.221216,6.0
116,A003,C037,St. Louis,MO,38.6270,-90.1994,0.315546,0.364623,0.272377,0.292044,...,0.153316,0.202021,0.140288,0.154836,0.186690,0.069620,0.166325,0.168310,16.761500,32.0
117,A003,C038,Milwaukee,WI,43.0389,-87.9065,0.441313,0.482022,0.546979,0.408705,...,0.593110,0.551330,0.508118,0.538109,0.658470,0.944304,0.631148,0.565827,58.868899,7.0
118,A003,C039,Richmond,VA,37.5407,-77.4360,0.183749,0.213398,0.158890,0.161169,...,0.112589,0.172356,0.180809,0.128691,0.124233,0.101266,0.142239,0.142740,14.256464,37.0


In [10]:
attendance_rate = 0.02  # assumption: 2% of a city's Spotify listeners will buy a ticket
 
# bring back the raw listener counts (index_output only has scaled scores)
index_output = index_output.merge(
    metrics_data[['artist_id', 'city_id', 'spotify_monthly_listeners']],
    on=['artist_id', 'city_id']
)

index_output['expected_demand'] = (index_output['spotify_monthly_listeners'] * attendance_rate).round()
index_output.sort_values(['artist_id', 'rank'])

,artist_id,city_id,city,state,lat,lon,spotify_monthly_listeners_log,streams_28d_log,instagram_likers_log,instagram_followers_log,...,youtube_viewers_28d_score,product_sales_units_score,song_sales_units_score,youtube_city_affinity_score,passive_score,proactive_score,city_index,rank,spotify_monthly_listeners,expected_demand
21,A001,C022,Portland,OR,45.5152,-122.6784,0.694124,0.691331,0.793888,0.649365,...,0.832834,0.956247,0.828333,1.000000,0.884878,0.879909,88.164797,1.0,234192,4684.0
2,A001,C003,Chicago,IL,41.8781,-87.6298,1.000000,0.987440,1.000000,1.000000,...,0.845608,0.705838,0.907735,0.863081,0.824424,0.808422,81.402265,2.0,610530,12211.0
12,A001,C013,Seattle,WA,47.6062,-122.3321,0.698367,0.719813,0.699882,0.638103,...,0.683100,0.574766,0.685857,0.915648,0.712765,0.610407,64.623257,3.0,237325,4746.0
10,A001,C011,Boston,MA,42.3601,-71.0589,0.770865,0.780760,0.711302,0.711649,...,0.627584,0.546790,0.748871,0.760391,0.701952,0.611277,64.301310,4.0,297835,5957.0
24,A001,C025,Austin,TX,30.2672,-97.7431,0.564064,0.547283,0.589672,0.572816,...,0.643630,0.468873,0.627890,0.743276,0.614416,0.567942,58.420823,5.0,155822,3116.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,A003,C034,Raleigh,NC,35.7796,-78.6382,0.215443,0.257024,0.232966,0.191551,...,0.170745,0.062198,0.130903,0.175949,0.175745,0.127159,14.416428,36.0,89384,1788.0
118,A003,C039,Richmond,VA,37.5407,-77.4360,0.183749,0.213398,0.158890,0.161169,...,0.180809,0.128691,0.124233,0.101266,0.142239,0.142740,14.256464,37.0,78099,1562.0
98,A003,C019,Orlando,FL,28.5383,-81.3792,0.283010,0.324505,0.285686,0.235215,...,0.069686,0.071320,0.102583,0.015190,0.125041,0.100347,10.898992,38.0,119183,2384.0
109,A003,C030,Indianapolis,IN,39.7684,-86.1581,0.197824,0.240008,0.176813,0.119555,...,0.101330,0.072168,0.065244,0.026582,0.089923,0.075191,8.034765,39.0,82923,1658.0


# Step 2: Identify candidate venues near each top city

In [11]:
venue_data = pd.read_csv('data/venues.csv').rename(columns={'metro_city_id': 'city_id'})
venue_data.head(5)

,venue_id,venue_name,venue_type,capacity,genres_booked,city_id,locality,state,season,lat,lon,avg_ticket_price_usd
0,V0001,Copper Room,club,350,country|jazz|folk rock,C001,New York,NY,year-round,40.76931,-74.01572,24
1,V0002,Echo Theater,theater,2950,metal|indie pop|r&b,C001,New York,NY,year-round,40.71045,-73.94402,67
2,V0003,Magnolia Playhouse,theater,1800,hip hop|jazz|r&b|metal,C001,New York,NY,year-round,40.70953,-74.00829,69
3,V0004,Echo Music Hall,music hall,4100,electronic|r&b|hip hop,C001,New York,NY,year-round,40.76338,-73.95350,60
4,V0005,Harbor Room,club,250,indie pop|rock,C001,New York,NY,year-round,40.68173,-73.99961,9


In [12]:
# top 10 cities per artist
## Step 1: filter to top 10 cities per artist and then merge with venue data

top_10_cities = index_output[index_output['rank'] <= 10].copy()
cand = top_10_cities[["artist_id", "city_id", "city", "lat", "lon", "expected_demand", "city_index"]].merge(
        venue_data, how="cross", suffixes=("_city", ""))
cand

,artist_id,city_id_city,city,lat_city,lon_city,expected_demand,city_index,venue_id,venue_name,venue_type,capacity,genres_booked,city_id,locality,state,season,lat,lon,avg_ticket_price_usd
0,A001,C001,New York,40.7128,-74.0060,10477.0,49.373287,V0001,Copper Room,club,350,country|jazz|folk rock,C001,New York,NY,year-round,40.76931,-74.01572,24
1,A001,C001,New York,40.7128,-74.0060,10477.0,49.373287,V0002,Echo Theater,theater,2950,metal|indie pop|r&b,C001,New York,NY,year-round,40.71045,-73.94402,67
2,A001,C001,New York,40.7128,-74.0060,10477.0,49.373287,V0003,Magnolia Playhouse,theater,1800,hip hop|jazz|r&b|metal,C001,New York,NY,year-round,40.70953,-74.00829,69
3,A001,C001,New York,40.7128,-74.0060,10477.0,49.373287,V0004,Echo Music Hall,music hall,4100,electronic|r&b|hip hop,C001,New York,NY,year-round,40.76338,-73.95350,60
4,A001,C001,New York,40.7128,-74.0060,10477.0,49.373287,V0005,Harbor Room,club,250,indie pop|rock,C001,New York,NY,year-round,40.68173,-73.99961,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9805,A003,C038,Milwaukee,43.0389,-87.9065,4677.0,58.868899,V0323,Golden Lounge,club,450,hip hop|country|r&b|folk rock,C040,Boise,ID,year-round,43.60294,-116.20509,44
9806,A003,C038,Milwaukee,43.0389,-87.9065,4677.0,58.868899,V0324,Copper Music Hall,music hall,4800,metal|jazz|folk rock,C040,Boise suburbs,ID,year-round,43.80188,-116.02508,74
9807,A003,C038,Milwaukee,43.0389,-87.9065,4677.0,58.868899,V0325,Riverside Ballroom,theater,2550,country|hip hop|folk rock,C040,Boise,ID,year-round,43.64366,-116.18377,53
9808,A003,C038,Milwaukee,43.0389,-87.9065,4677.0,58.868899,V0326,Harbor Social Club,club,750,jazz|r&b|hip hop|metal|folk rock,C040,Boise,ID,year-round,43.59414,-116.15049,33


In [13]:
# step 2: keep only venues close to the city
radius_miles = 10  # default radius for decision making

def haversine_miles(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 3958.8 * 2 * np.arcsin(np.sqrt(a))   # 3958.8 = Earth's radius in miles

cand["distance_miles"] = haversine_miles(cand.lat_city, cand.lon_city, cand.lat, cand.lon).round(1)
cand = cand[cand.distance_miles <= radius_miles]
cand

,artist_id,city_id_city,city,lat_city,lon_city,expected_demand,city_index,venue_id,venue_name,venue_type,capacity,genres_booked,city_id,locality,state,season,lat,lon,avg_ticket_price_usd,distance_miles
0,A001,C001,New York,40.7128,-74.0060,10477.0,49.373287,V0001,Copper Room,club,350,country|jazz|folk rock,C001,New York,NY,year-round,40.76931,-74.01572,24,3.9
1,A001,C001,New York,40.7128,-74.0060,10477.0,49.373287,V0002,Echo Theater,theater,2950,metal|indie pop|r&b,C001,New York,NY,year-round,40.71045,-73.94402,67,3.3
2,A001,C001,New York,40.7128,-74.0060,10477.0,49.373287,V0003,Magnolia Playhouse,theater,1800,hip hop|jazz|r&b|metal,C001,New York,NY,year-round,40.70953,-74.00829,69,0.3
3,A001,C001,New York,40.7128,-74.0060,10477.0,49.373287,V0004,Echo Music Hall,music hall,4100,electronic|r&b|hip hop,C001,New York,NY,year-round,40.76338,-73.95350,60,4.4
4,A001,C001,New York,40.7128,-74.0060,10477.0,49.373287,V0005,Harbor Room,club,250,indie pop|rock,C001,New York,NY,year-round,40.68173,-73.99961,9,2.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9456,A003,C036,Detroit,42.3314,-83.0458,11432.0,59.221216,V0301,Blue Theater,theater,2350,country|metal|rock,C036,Detroit,MI,year-round,42.34612,-83.09431,48,2.7
9457,A003,C036,Detroit,42.3314,-83.0458,11432.0,59.221216,V0302,Blue Playhouse,theater,2050,indie pop|country|folk rock|hip hop,C036,Detroit,MI,year-round,42.36897,-83.11249,69,4.3
9792,A003,C038,Milwaukee,43.0389,-87.9065,4677.0,58.868899,V0310,Blue Pavilion Hall,music hall,3100,r&b|hip hop|electronic,C038,Milwaukee,WI,year-round,43.00297,-87.91652,73,2.5
9793,A003,C038,Milwaukee,43.0389,-87.9065,4677.0,58.868899,V0311,Ironworks Room,club,600,metal|jazz,C038,Milwaukee,WI,year-round,43.03632,-87.92457,16,0.9


In [14]:
cand["capacity_fit"] = cand.capacity / cand.expected_demand
cand = cand[cand.capacity_fit.between(0.5, 1.5)]
cand

,artist_id,city_id_city,city,lat_city,lon_city,expected_demand,city_index,venue_id,venue_name,venue_type,...,genres_booked,city_id,locality,state,season,lat,lon,avg_ticket_price_usd,distance_miles,capacity_fit
342,A001,C002,Los Angeles,34.0522,-118.2437,9595.0,55.983047,V0016,Granite Arena,arena,...,jazz|country|hip hop|rock|indie pop,C002,Los Angeles,CA,year-round,34.04573,-118.20147,125,2.5,1.412194
343,A001,C002,Los Angeles,34.0522,-118.2437,9595.0,55.983047,V0017,Golden Amphitheater,amphitheater,...,rock|country|jazz|hip hop,C002,Los Angeles,CA,May-Oct,34.06623,-118.26990,100,1.8,1.021365
353,A001,C002,Los Angeles,34.0522,-118.2437,9595.0,55.983047,V0027,Blue Pavilion Hall,music hall,...,r&b|jazz|indie pop|hip hop,C002,Los Angeles,CA,year-round,34.07040,-118.16829,66,4.5,0.552371
354,A001,C002,Los Angeles,34.0522,-118.2437,9595.0,55.983047,V0028,Summit Bowl,amphitheater,...,indie pop|r&b|hip hop,C002,Los Angeles,CA,May-Oct,34.06190,-118.27621,88,2.0,0.901511
682,A001,C003,Chicago,41.8781,-87.6298,12211.0,81.402265,V0029,Magnolia Coliseum,arena,...,country|metal|folk rock|hip hop,C003,Chicago,IL,year-round,41.81777,-87.65178,111,4.3,1.179265
692,A001,C003,Chicago,41.8781,-87.6298,12211.0,81.402265,V0039,Northside Bowl,amphitheater,...,r&b|rock|indie pop|folk rock,C003,Chicago,IL,May-Oct,41.87592,-87.61949,76,0.6,0.683810
1097,A001,C011,Boston,42.3601,-71.0589,5957.0,64.301310,V0117,Echo Music Hall,music hall,...,folk rock|jazz|country|hip hop|indie pop,C011,Boston,MA,year-round,42.35975,-71.05896,72,0.0,0.814168
1099,A001,C011,Boston,42.3601,-71.0589,5957.0,64.301310,V0119,Juniper Pavilion Hall,music hall,...,jazz|rock|metal|indie pop,C011,Boston,MA,year-round,42.37357,-70.97111,88,4.6,0.587544
1444,A001,C013,Seattle,47.6062,-122.3321,4746.0,64.623257,V0137,Riverside Auditorium,music hall,...,jazz|indie pop|electronic|folk rock,C013,Seattle,WA,year-round,47.60479,-122.32680,50,0.3,1.074589
1446,A001,C013,Seattle,47.6062,-122.3321,4746.0,64.623257,V0139,Union Amphitheater,amphitheater,...,electronic|metal|folk rock|country,C013,Seattle,WA,May-Oct,47.56834,-122.25222,76,4.5,1.369574


In [15]:
## this is a new column that counts the number of open dates for each venue (made up dates)
start_date = '2027-03-01'  # first day of the tour window
end_date = '2027-06-30'    # last day of the tour window

window = pd.DataFrame({"event_date": pd.date_range(start_date, end_date, freq="D")})
window

,event_date
0,2027-03-01
1,2027-03-02
2,2027-03-03
3,2027-03-04
4,2027-03-05
...,...
117,2027-06-26
118,2027-06-27
119,2027-06-28
120,2027-06-29


In [16]:
grid = cand[["venue_id"]].drop_duplicates().merge(window, how="cross")
grid 

,venue_id,event_date
0,V0016,2027-03-01
1,V0016,2027-03-02
2,V0016,2027-03-03
3,V0016,2027-03-04
4,V0016,2027-03-05
...,...,...
6339,V0314,2027-06-26
6340,V0314,2027-06-27
6341,V0314,2027-06-28
6342,V0314,2027-06-29


In [ ]:
events = pd.read_csv('data/scheduled_events.csv', parse_dates=['event_date'])


booked = events[["venue_id", "event_date"]].assign(booked=True)
booked["event_date"] = pd.to_datetime(booked["event_date"])   # make sure it is a real date, not text, so it matches the calendar grid
booked

,venue_id,event_date,booked
0,V0001,2027-03-01,True
1,V0001,2027-03-02,True
2,V0001,2027-03-03,True
3,V0001,2027-03-05,True
4,V0001,2027-03-06,True
...,...,...,...
19517,V0327,2027-06-25,True
19518,V0327,2027-06-26,True
19519,V0327,2027-06-27,True
19520,V0327,2027-06-28,True


In [18]:
grid = grid.merge(booked, on=["venue_id", "event_date"], how="left")
grid 

,venue_id,event_date,booked
0,V0016,2027-03-01,NaN
1,V0016,2027-03-02,True
2,V0016,2027-03-03,NaN
3,V0016,2027-03-04,NaN
4,V0016,2027-03-05,True
...,...,...,...
6339,V0314,2027-06-26,True
6340,V0314,2027-06-27,True
6341,V0314,2027-06-28,True
6342,V0314,2027-06-29,True


In [19]:
# keep only the free days (calendar minus booked days)
open_dates = grid[grid.booked.isna()].copy()
open_dates

,venue_id,event_date,booked
0,V0016,2027-03-01,NaN
2,V0016,2027-03-03,NaN
3,V0016,2027-03-04,NaN
5,V0016,2027-03-06,NaN
8,V0016,2027-03-09,NaN
...,...,...,...
6334,V0314,2027-06-21,NaN
6335,V0314,2027-06-22,NaN
6336,V0314,2027-06-23,NaN
6337,V0314,2027-06-24,NaN


In [20]:
# outdoor venues (season = May-Oct) are closed the rest of the year, so remove those days
seasonal = venue_data.loc[venue_data.season == "May-Oct", "venue_id"]

is_seasonal_venue = open_dates.venue_id.isin(seasonal)
is_in_season = open_dates.event_date.dt.month.between(5, 10)

open_dates = open_dates[~(is_seasonal_venue & ~is_in_season)]
open_dates

,venue_id,event_date,booked
0,V0016,2027-03-01,NaN
2,V0016,2027-03-03,NaN
3,V0016,2027-03-04,NaN
5,V0016,2027-03-06,NaN
8,V0016,2027-03-09,NaN
...,...,...,...
6334,V0314,2027-06-21,NaN
6335,V0314,2027-06-22,NaN
6336,V0314,2027-06-23,NaN
6337,V0314,2027-06-24,NaN


In [21]:
# flag weekend days (dayofweek: Mon=0 ... Fri=4, Sat=5, Sun=6, so >= 4 means Fri-Sun)
open_dates["is_weekend"] = open_dates.event_date.dt.dayofweek >= 4
open_dates

,venue_id,event_date,booked,is_weekend
0,V0016,2027-03-01,NaN,False
2,V0016,2027-03-03,NaN,False
3,V0016,2027-03-04,NaN,False
5,V0016,2027-03-06,NaN,True
8,V0016,2027-03-09,NaN,False
...,...,...,...,...
6334,V0314,2027-06-21,NaN,False
6335,V0314,2027-06-22,NaN,False
6336,V0314,2027-06-23,NaN,False
6337,V0314,2027-06-24,NaN,False


In [22]:
# summarize availability for each venue
avail = open_dates.groupby("venue_id").agg(
    open_dates=("event_date", "count"),                  # how many free days in total
    open_weekend_dates=("is_weekend", "sum"),            # how many of them are Fri-Sun
    first_open_weekend=("event_date", lambda d: d[d.dt.dayofweek >= 4].min()),   # earliest free weekend day
).reset_index()
avail

,venue_id,open_dates,open_weekend_dates,first_open_weekend
0,V0002,44,11,2027-03-13
1,V0003,55,14,2027-03-06
2,V0007,76,25,2027-03-05
3,V0011,48,9,2027-03-05
4,V0012,64,16,2027-03-07
5,V0016,75,27,2027-03-06
6,V0017,41,15,2027-05-01
7,V0018,55,12,2027-03-07
8,V0019,91,32,2027-03-05
9,V0020,69,17,2027-03-05


In [23]:
# attach the availability summary to the candidate venues
# (inner merge: a venue with zero free days drops out here)
shortlist = cand.merge(avail, on="venue_id")
shortlist

,artist_id,city_id_city,city,lat_city,lon_city,expected_demand,city_index,venue_id,venue_name,venue_type,...,state,season,lat,lon,avg_ticket_price_usd,distance_miles,capacity_fit,open_dates,open_weekend_dates,first_open_weekend
0,A001,C002,Los Angeles,34.0522,-118.2437,9595.0,55.983047,V0016,Granite Arena,arena,...,CA,year-round,34.04573,-118.20147,125,2.5,1.412194,75,27,2027-03-06
1,A001,C002,Los Angeles,34.0522,-118.2437,9595.0,55.983047,V0017,Golden Amphitheater,amphitheater,...,CA,May-Oct,34.06623,-118.26990,100,1.8,1.021365,41,15,2027-05-01
2,A001,C002,Los Angeles,34.0522,-118.2437,9595.0,55.983047,V0027,Blue Pavilion Hall,music hall,...,CA,year-round,34.07040,-118.16829,66,4.5,0.552371,76,25,2027-03-06
3,A001,C002,Los Angeles,34.0522,-118.2437,9595.0,55.983047,V0028,Summit Bowl,amphitheater,...,CA,May-Oct,34.06190,-118.27621,88,2.0,0.901511,44,14,2027-05-01
4,A001,C003,Chicago,41.8781,-87.6298,12211.0,81.402265,V0029,Magnolia Coliseum,arena,...,IL,year-round,41.81777,-87.65178,111,4.3,1.179265,66,16,2027-03-12
5,A001,C003,Chicago,41.8781,-87.6298,12211.0,81.402265,V0039,Northside Bowl,amphitheater,...,IL,May-Oct,41.87592,-87.61949,76,0.6,0.683810,53,20,2027-05-01
6,A001,C011,Boston,42.3601,-71.0589,5957.0,64.301310,V0117,Echo Music Hall,music hall,...,MA,year-round,42.35975,-71.05896,72,0.0,0.814168,66,18,2027-03-12
7,A001,C011,Boston,42.3601,-71.0589,5957.0,64.301310,V0119,Juniper Pavilion Hall,music hall,...,MA,year-round,42.37357,-70.97111,88,4.6,0.587544,73,20,2027-03-06
8,A001,C013,Seattle,47.6062,-122.3321,4746.0,64.623257,V0137,Riverside Auditorium,music hall,...,WA,year-round,47.60479,-122.32680,50,0.3,1.074589,68,21,2027-03-05
9,A001,C013,Seattle,47.6062,-122.3321,4746.0,64.623257,V0139,Union Amphitheater,amphitheater,...,WA,May-Oct,47.56834,-122.25222,76,4.5,1.369574,40,14,2027-05-01


In [24]:
# score each venue on three things, each between 0 and 1
# capacity: 1.0 when the room size matches expected demand exactly
shortlist["capacity_score"] = 1 - (shortlist.capacity_fit - 1).abs()

# flexibility: more free weekend days is better (min-max scaled within each artist)
shortlist["flexibility_score"] = shortlist.groupby("artist_id")["open_weekend_dates"].transform(min_max_normalization)

# distance score: 1.0 at the city center, 0.0 at the edge of the radius
shortlist["distance_score"] = 1 - shortlist.distance_miles / radius_miles

shortlist[["artist_id", "city", "venue_name", "capacity_fit", "capacity_score",
           "open_weekend_dates", "flexibility_score", "distance_miles", "distance_score"]]

,artist_id,city,venue_name,capacity_fit,capacity_score,open_weekend_dates,flexibility_score,distance_miles,distance_score
0,A001,Los Angeles,Granite Arena,1.412194,0.587806,27,0.777778,2.5,0.75
1,A001,Los Angeles,Golden Amphitheater,1.021365,0.978635,15,0.333333,1.8,0.82
2,A001,Los Angeles,Blue Pavilion Hall,0.552371,0.552371,25,0.703704,4.5,0.55
3,A001,Los Angeles,Summit Bowl,0.901511,0.901511,14,0.296296,2.0,0.80
4,A001,Chicago,Magnolia Coliseum,1.179265,0.820735,16,0.370370,4.3,0.57
5,A001,Chicago,Northside Bowl,0.683810,0.683810,20,0.518519,0.6,0.94
6,A001,Boston,Echo Music Hall,0.814168,0.814168,18,0.444444,0.0,1.00
7,A001,Boston,Juniper Pavilion Hall,0.587544,0.587544,20,0.518519,4.6,0.54
8,A001,Seattle,Riverside Auditorium,1.074589,0.925411,21,0.555556,0.3,0.97
9,A001,Seattle,Union Amphitheater,1.369574,0.630426,14,0.296296,4.5,0.55


In [ ]:
# combine the three scores into one venue_score
shortlist["venue_score"] = (
    0.5 * shortlist.capacity_score
    + 0.3 * shortlist.flexibility_score
    + 0.2 * shortlist.distance_score
).round(3)

# best cities first, and best venue first within each city
shortlist = shortlist.sort_values(["artist_id", "city_index", "venue_score"], ascending=[True, False, False])
shortlist["first_open_weekend"] = shortlist.first_open_weekend.dt.date

out_cols = ["artist_id", "city", "venue_name", "venue_type", "locality", "capacity", "expected_demand",
            "capacity_fit", "distance_miles", "open_dates", "open_weekend_dates",
            "first_open_weekend", "avg_ticket_price_usd", "venue_score"]
shortlist[out_cols].round(2)

,artist_id,city,venue_name,venue_type,locality,capacity,expected_demand,capacity_fit,distance_miles,open_dates,open_weekend_dates,first_open_weekend,avg_ticket_price_usd,venue_score
14,A001,Portland,Echo Music Hall,music hall,Portland,5000,4684.0,1.07,3.9,69,23,2027-03-13,72,0.78
15,A001,Portland,Velvet Auditorium,music hall,Portland,4800,4684.0,1.02,1.4,56,12,2027-03-12,76,0.73
5,A001,Chicago,Northside Bowl,amphitheater,Chicago,8350,12211.0,0.68,0.6,53,20,2027-05-01,76,0.68
4,A001,Chicago,Magnolia Coliseum,arena,Chicago,14400,12211.0,1.18,4.3,66,16,2027-03-12,111,0.64
8,A001,Seattle,Riverside Auditorium,music hall,Seattle,5100,4746.0,1.07,0.3,68,21,2027-03-05,50,0.82
9,A001,Seattle,Union Amphitheater,amphitheater,Seattle,6500,4746.0,1.37,4.5,40,14,2027-05-01,76,0.51
6,A001,Boston,Echo Music Hall,music hall,Boston,4850,5957.0,0.81,0.0,66,18,2027-03-12,72,0.74
7,A001,Boston,Juniper Pavilion Hall,music hall,Boston,3500,5957.0,0.59,4.6,73,20,2027-03-06,88,0.56
17,A001,Austin,Harbor Pavilion Hall,music hall,Austin,3950,3116.0,1.27,0.2,72,25,2027-03-07,77,0.77
16,A001,Austin,Harbor Playhouse,theater,Austin,2050,3116.0,0.66,4.7,63,23,2027-03-05,56,0.62


In [26]:
# the single best venue for each artist in each city
best = shortlist.groupby(["artist_id", "city"], sort=False).head(1)
best[out_cols].round(2)

,artist_id,city,venue_name,venue_type,locality,capacity,expected_demand,capacity_fit,distance_miles,open_dates,open_weekend_dates,first_open_weekend,avg_ticket_price_usd,venue_score
14,A001,Portland,Echo Music Hall,music hall,Portland,5000,4684.0,1.07,3.9,69,23,2027-03-13,72,0.78
5,A001,Chicago,Northside Bowl,amphitheater,Chicago,8350,12211.0,0.68,0.6,53,20,2027-05-01,76,0.68
8,A001,Seattle,Riverside Auditorium,music hall,Seattle,5100,4746.0,1.07,0.3,68,21,2027-03-05,50,0.82
6,A001,Boston,Echo Music Hall,music hall,Boston,4850,5957.0,0.81,0.0,66,18,2027-03-12,72,0.74
17,A001,Austin,Harbor Pavilion Hall,music hall,Austin,3950,3116.0,1.27,0.2,72,25,2027-03-07,77,0.77
10,A001,Minneapolis,Velvet Music Hall,music hall,Minneapolis,4100,3898.0,1.05,0.1,80,21,2027-03-06,69,0.84
1,A001,Los Angeles,Golden Amphitheater,amphitheater,Los Angeles,9800,9595.0,1.02,1.8,41,15,2027-05-01,100,0.75
13,A001,Orlando,Granite Theater,theater,Orlando,2200,2881.0,0.76,1.2,52,6,2027-04-09,48,0.56
19,A001,St. Louis,Magnolia Theater,theater,St. Louis,2350,2796.0,0.84,4.2,85,33,2027-03-06,62,0.84
39,A002,Austin,Neon Social Club,club,Austin,800,879.0,0.91,1.7,48,8,2027-03-12,40,0.66


In [30]:
# which top-10 cities ended up with no venue at all?
all_pairs = set(zip(top_10_cities.artist_id, top_10_cities.city))
found_pairs = set(zip(best.artist_id, best.city))
sorted(all_pairs - found_pairs)   # try a larger radius_miles or a wider capacity_fit band for these

[('A001', 'New York'),
 ('A002', 'Salt Lake City'),
 ('A003', 'Minneapolis'),
 ('A003', 'New York'),
 ('A003', 'San Diego'),
 ('A003', 'Washington')]

### Save the results for Step 3
#### Starting from here entire work done by AI

Step 3 (`Plan Logistics.ipynb`) needs two things from this notebook: the best venue for each artist in each
city, and the weekend dates that venue still has open.

In [28]:
# save the best venue per artist/city and the open weekend dates for the next notebook
import os
os.makedirs('output', exist_ok=True)

best_venues = best[['artist_id', 'city_id', 'city', 'state', 'venue_id', 'venue_name', 'venue_type', 'locality',
                    'capacity', 'expected_demand', 'capacity_fit', 'lat', 'lon', 'distance_miles',
                    'avg_ticket_price_usd', 'city_index', 'venue_score']].rename(columns={'lat': 'venue_lat', 'lon': 'venue_lon'})
best_venues.to_csv('output/step2_best_venues.csv', index=False)

open_weekend_dates = open_dates[open_dates.is_weekend & open_dates.venue_id.isin(best_venues.venue_id)][['venue_id', 'event_date']]
open_weekend_dates.to_csv('output/step2_open_weekend_dates.csv', index=False)

best_venues.shape, open_weekend_dates.shape

((24, 17), (463, 2))

### Export Steps 1-2 for the app

The app (`app/index.html`) shows each step. This cell saves the city scores, the metrics behind them, and the
venue shortlist to `app/index_data.js`.

In [29]:
# save everything the app needs to show Steps 1 and 2 (app/index_data.js)
import json, os

# the 8 metrics: key, label, factor, is it a raw count? (youtube_city_affinity is already an index)
metric_info = [
    ('spotify_monthly_listeners', 'Spotify monthly listeners', 'Passive Engagement', True),
    ('youtube_city_affinity', 'YouTube city affinity', 'Passive Engagement', False),
    ('streams_28d', 'Streams (28 days)', 'Passive Engagement', True),
    ('instagram_likers', 'Instagram likers', 'Proactive Engagement', True),
    ('instagram_followers', 'Instagram followers', 'Proactive Engagement', True),
    ('youtube_viewers_28d', 'YouTube viewers (28 days)', 'Proactive Engagement', True),
    ('product_sales_units', 'Product sales (units)', 'Proactive Engagement', True),
    ('song_sales_units', 'Song sales (units)', 'Proactive Engagement', True),
]
raw_cols = [m[0] for m in metric_info]
r4 = lambda x: round(float(x), 4)

# Step 1: one row per artist x city, with the raw metrics and every scaled score
city_table = index_output.drop(columns=['spotify_monthly_listeners']).merge(
    metrics_data[['artist_id', 'city_id', 'metro_population'] + raw_cols], on=['artist_id', 'city_id'])

# Step 2: how many venue/city pairs survive each filter (the funnel), per artist and city
pairs = top_10_cities[["artist_id", "city_id", "city", "lat", "lon", "expected_demand", "city_index"]].merge(
    venue_data, how="cross", suffixes=("_city", ""))
pairs['distance_miles'] = haversine_miles(pairs.lat_city, pairs.lon_city, pairs.lat, pairs.lon)
pairs['within'] = pairs.distance_miles <= radius_miles
pairs['right_size'] = pairs.within & (pairs.capacity / pairs.expected_demand).between(0.5, 1.5)
pairs['has_open_dates'] = pairs.right_size & pairs.venue_id.isin(avail.venue_id)
funnel = pairs.groupby(['artist_id', 'city']).agg(
    all=('venue_id', 'size'), within=('within', 'sum'), right_size=('right_size', 'sum'), open=('has_open_dates', 'sum'))

# does the venue book the artist's genre? (shown in the app for information, it is not a filter yet)
genre = artist_data.set_index('artist_id').primary_genre
# the free weekend dates of every venue (the app re-plans routes in the browser)
weekend_dates = open_dates[open_dates.is_weekend].groupby('venue_id').event_date.apply(
    lambda d: [x.date().isoformat() for x in sorted(d)]).to_dict()
best_keys = set(zip(best.artist_id, best.venue_id))

artists_out = []
for a in artist_data.itertuples():
    cities_out = []
    for c in city_table[city_table.artist_id == a.artist_id].sort_values('rank').itertuples():
        row = c._asdict()
        cities_out.append({
            'city_id': c.city_id, 'city': c.city, 'state': c.state, 'rank': int(c.rank), 'city_index': r4(c.city_index),
            'passive_score': r4(c.passive_score), 'proactive_score': r4(c.proactive_score),
            'metro_population': int(c.metro_population), 'expected_demand': int(c.expected_demand),
            'raw': {k: float(row[k]) for k in raw_cols},
            'vol': {k: r4(row[k + '_log']) for k, *_, is_count in metric_info if is_count},
            'per': {k: r4(row[k + '_penetration']) for k, *_, is_count in metric_info if is_count},
            'score': {k: r4(row[k + '_score']) for k in raw_cols},
        })
    venues_out = {}
    for v in shortlist[shortlist.artist_id == a.artist_id].sort_values('venue_score', ascending=False).itertuples():
        venues_out.setdefault(v.city, []).append({
            'venue_id': v.venue_id, 'venue': v.venue_name, 'type': v.venue_type, 'locality': v.locality,
            'city_id': v.city_id, 'state': v.state, 'lat': float(v.lat), 'lon': float(v.lon),
            'weekend_dates': weekend_dates.get(v.venue_id, []),
            'capacity': int(v.capacity), 'capacity_fit': r4(v.capacity_fit), 'distance_miles': float(v.distance_miles),
            'open_dates': int(v.open_dates), 'open_weekend_dates': int(v.open_weekend_dates),
            'first_open_weekend': str(v.first_open_weekend), 'avg_ticket_price_usd': int(v.avg_ticket_price_usd),
            'genres': v.genres_booked.split('|'), 'books_genre': a.primary_genre in v.genres_booked.split('|'),
            'capacity_score': r4(v.capacity_score), 'flexibility_score': r4(v.flexibility_score),
            'distance_score': r4(v.distance_score), 'venue_score': r4(v.venue_score),
            'is_best': (a.artist_id, v.venue_id) in best_keys,
        })
    f = funnel.loc[a.artist_id]
    artists_out.append({
        'artist_id': a.artist_id, 'artist_name': a.artist_name, 'genre': a.primary_genre,
        'cities': cities_out, 'venues': venues_out,
        'funnel': {city: {k: int(f.loc[city, k]) for k in ['all', 'within', 'right_size', 'open']} for city in f.index},
    })

index_data = {
    'params': {'attendance_rate': attendance_rate, 'radius_miles': radius_miles, 'capacity_band': [0.5, 1.5],
               'window': {'start': start_date, 'end': end_date},
               'factor_weights': factor_weights},
    'metrics': [{'key': k, 'label': label, 'factor': factor, 'count': is_count,
                 'weight': variable_weights[factor][k + '_score']} for k, label, factor, is_count in metric_info],
    'artists': artists_out,
}
os.makedirs('app', exist_ok=True)
with open('app/index_data.js', 'w') as fh:
    fh.write('window.INDEX_DATA = ' + json.dumps(index_data) + ';')

[(a['artist_name'], len(a['cities']), sum(len(v) for v in a['venues'].values())) for a in artists_out]

[('Nova Reyes', 40, 21), ('The Copper Pines', 40, 22), ('Kairo Blaze', 40, 10)]